# Swapcard Exhibitor Link Parser

This parses copied Swapcard/Hardware Pioneers exhibitor HTML.

It extracts:
- company name
- stand/location
- exhibitor profile link
- logo proxy URL
- original logo URL

Important:
If the webpage has a "Load more" button, your saved HTML only contains the exhibitors currently loaded.
Click "Load more" repeatedly first, then copy/save the page HTML again.



## 1. Imports



In [ ]:
import csv
import re
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse

from bs4 import BeautifulSoup



## 2. Choose HTML Input

Option A: put your saved/copied HTML into a `.txt` or `.html` file and set `HTML_FILE`.

Option B: paste HTML directly into `HTML_TEXT`.



In [ ]:
HTML_FILE = None
HTML_TEXT = r"""
<div class="sc-epgtWL gBSvwX"><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTQ="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Esprit Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F101595db350b41fbbda19da2f64928fd.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Esprit Electronics</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand F9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjYyNTE="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="PCBWay" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F08061b680d7c434ab6411bcf72358955.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">PCBWay</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand S10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTA="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Win Source" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F556a2029128043ffa2778694ed26da28.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Win Source</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand N5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDY="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="2J Antennas &amp; Antenova" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd2cde975000e40e6ba1afcb13d5b70f2.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">2J Antennas &amp; Antenova</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand G2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDQ="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="3Point1 Developments" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7708c088f0684745b231d37ece8749a1.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">3Point1 Developments</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand D11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDU="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="AAV Plastics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F3fb16d1ecf0e4597b09aa71f4dfd93fc.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">AAV Plastics</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand A11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTM="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Abracon" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F455b69f7dcea4e1fbc58a524b07d2946.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Abracon</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand D6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTQ="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Active-PCB" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7b8552fbadb140698952cd80e166ee07.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Active-PCB</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand M12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NzQ1NjI="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Alif Semiconductor" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F78db2101e3f1489b96e9a4e084d2d990.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Alif Semiconductor</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand H6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTY="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Alliot" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7f5b898edad5433799961243ba14d8c8.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Alliot</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand B10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTg="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Alpha Micro Components" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fe072b7d938324866bc6d2a6f0eab03fb.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Alpha Micro Components</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand N6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MjM="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Altium" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fb728d88f74f343d6ab4826dc5083f66a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Altium</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand N7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjU4ODY="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Ambient Scientific" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F962b0d65281a4c7297c459518ad90a0e.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Ambient Scientific</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Sponsor Only</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MzcwMzI="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Analog Devices" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fb44fa2e97e8b4fa686a39bc8565f6bec.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Analog Devices</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand N4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMTQ0NDg="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Anglia Live" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc94f02c2a46e4612bb7e304ecaf5d2af.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Anglia Live</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand H8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MjQ="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Ansmann UK" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fcdacb924288844e5aed566c68bdad875.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Ansmann UK</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand Q15</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0Njk3ODk="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Apitronix Semiconductor" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd75d06fcceb24c7692ccd023ee7c59d5.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Apitronix Semiconductor</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Startup Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MjI="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Ardencraft Technology" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F30aa1bc5a8d34912886356cf0c45d996.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Ardencraft Technology</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand G3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MzA="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Arrow" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F86ebf1fcc22a4b069b0aefdcf878ec63.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Arrow</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand L8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Mjg="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="ASK Technology" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9761bd6b98e14c06921181eabae1b121.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">ASK Technology</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand A18</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Mjk="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Astute Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F09928afd42584da1bad6c140c167bb14.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Astute Electronics</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand H6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MzQ="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Asunny Circuits" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc7dfba9982184da8b848a575034d7181.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Asunny Circuits</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand A7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTk="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Atlas Procurement Solution" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F8e8d0a1dc8be477d8f09494a14106321.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Atlas Procurement Solution</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand C9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NTcwODU="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="AutoPCB" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F4203200e27d34c828dddda4b297be4d1.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">AutoPCB</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand G6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MzY="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Avnet Abacus" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F61d511eda1804086bd37657236b8a132.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Avnet Abacus</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand H1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MzU="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Avnet Silica" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fea8d9b7757094518a4010c3682e77aa9.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Avnet Silica</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand H2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NDE="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Axiomtek" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fbb53d083b05c4757976e77f7f0b3cecf.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Axiomtek</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand B7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NDA="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Azego TS Ltd and Vision Display Solutions" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ff2743cdd59074df2b53ffae2db9b9d05.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Azego TS Ltd and Vision Display Solutions</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand H5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NDI="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="BCD Atlantik" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd325edd116e44b04b8b03ca556e41737.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">BCD Atlantik</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand D1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjAxODM="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="BEC Group" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fa4824251f7b74de48b3de994531d12d0.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">BEC Group</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand S8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NDY="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Binder" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F72c2c03747f24f1da2a77548392b55df.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Binder</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand F11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NDc="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Bittium" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F33cd30847989427296be0be43ce2986a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Bittium</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand B15</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NTI="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Bloomice" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F0d43fd9a3a9444b196f64982f805d64a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Bloomice</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand G4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzMDI="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Bodio Electronic Technologies" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F10d5b28808524f78b22d7437c96146b7.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Bodio Electronic Technologies</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand S6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTI="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Braemac" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F64a65a1a0146499797f85d146b58a0c6.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Braemac</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand G11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NTM="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Brainboxes" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F86706d8bea97414ba94928593045239a.jpeg&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Brainboxes</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand Q14</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NTQ="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="BSI Group" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ff1049f10ea494280b1d3838f140a433c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">BSI Group</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand A15</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTM="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Bulgin" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F07a7b3e221f340dc8323cced672a5b8a.jpeg&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Bulgin</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand C6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NTY="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="ByteSnap Design" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F823672fe43964bc1bd5fdcfcd53a6c07.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">ByteSnap Design</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand C2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjAyMjE="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="C-MAC Electronics Solutions" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F59f9c99112854dc8ad7bc7e6c5508287.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">C-MAC Electronics Solutions</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand S1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NTg3NzM="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Caligra" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F6847ff1a700543d787b246c3041641e8.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Caligra</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand A12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NTU="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Cambridge Electronic Industries" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F3c3576f61f914d6380fbbfc809f78b53.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Cambridge Electronic Industries</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand A3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMTQ0NDQ="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Challenger Solutions" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F331446c34f4e4287abcc7d8fc2fa9aac.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Challenger Solutions</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand C16</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjQxOTU="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Charcroft Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fa48174247ca94ca3ac13cba6983df75a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Charcroft Electronics</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand A14</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0Njk3OTk="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Chevin Technology" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F605521a9a4f2433e98131f6bbf2eb1ba.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Chevin Technology</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Startup Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0Njk2Njc="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Chipletti" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Facf74251332947838b724a9680f054d5.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Chipletti</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Startup Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDM="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Cicor Group" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fabf3e4ca84d7467c925852cb90df287e.jpeg&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Cicor Group</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand P1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDUwMjY="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Circutor" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fee949d5545a544498e7f34f1d5f0e984.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Circutor</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand S5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMTQ0NDY="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="COAX Connectors" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fa4c88769154c4eca97fbf8bd1dfcf302.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">COAX Connectors</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand A24</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NTc="><div class="sc-hBFaCF cnkThY sc-iODBqg hTSlDd"><div class="sc-cYKLtW kdDIcH"><div class="sc-clQjiH cvvmVs sc-eWuiuV jLfEQc" size="224"><img size="224" ratio="2" class="sc-bMlmE gOHjTX" alt="Coilcraft Europe" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9c91f3d349084ab5a6ac13330ebc40d4.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-dcyarI hFxWCA"><div class="sc-ilnmmj cSWbBy"><span class="sc-iBaQfa sc-iWqcWD ehlYHy cnMTJl">Coilcraft Europe</span><span class="sc-iBaQfa sc-elrrKB dBDoXm bpNmVH">Stand G7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NjE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Consult Red" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fb299d141ce0c436cb1b724996425eb3f.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Consult Red</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">VIP &amp; Press Lounge</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NjI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Cornelius Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F53692c3254674d9b9bf2d23e94aebea3.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Cornelius Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NjM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Crank AMETEK" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F85f90db8534e46219a04cdb494db7c20.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Crank AMETEK</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Njc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Datalink Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F66c64afadc9b4279a2a40f7d37d58052.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Datalink Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDc4NDk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="DAU Components" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F66880798166c4b1dbb4cd7711fd8d90d.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">DAU Components</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand S7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NjY1ODM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="DeepGate AI" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F6824b46852cc4aff8247c87623a12a8b.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">DeepGate AI</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Njg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="DeepX" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F99ac398259b64f189f86e46ea4c34fb1.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">DeepX</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sponsor Only</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Njk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Diamond Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fdb7d07a194664b7e8954dcaa68225149.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Diamond Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NzQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Digi International" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7e018948396645498bd801503cd6859e.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Digi International</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NzM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="DigiKey" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fe7eda013f02d48939f89afc7ebed8ae1.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">DigiKey</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sponsor Only</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NTMyMjM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Diodes Incorporated" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F6084653346794bda98d36c76928b71bd.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Diodes Incorporated</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDQxNzA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Direct Insight" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc9c51703f8d44288b41b588986343074.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Direct Insight</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sponsor Only</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="DMTL" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd551b31c7da14a8197b2e82a16dadb4d.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">DMTL</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Nzk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="DPTechnics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F5c38d25343ac44aba826dd7f6875f6a1.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">DPTechnics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzMjg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="E-Switch" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fe55ea8ee139c40aca59aab84c9d9a546.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">E-Switch</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand S3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="E-tec Interconnect" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F37188134a17d46ca9459677e3f107cc8.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">E-tec Interconnect</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Easby Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F36cd25bd43244a6b96a3a1b70bc51e27.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Easby Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzODE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Eaton" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F35d0c8aca2854809bcb3cf1f00e9e39b.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Eaton</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="EBV Elektronik GmbH" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F5dda51aad9544024adce2a4889aa3307.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">EBV Elektronik GmbH</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F15</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMTM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="EC Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ffa2a267c7abe4e199251d75563876832.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">EC Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDM5NzY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Ecopac Power" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F1ea480119f2044d394394bf84fb1c8d3.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Ecopac Power</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjcyOTA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Edge AI Foundation" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd0eb9b2f085545df989ffa5dc6b0b561.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Edge AI Foundation</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sponsor Only</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMDM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Edge Impulse a Qualcomm company" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F042ff12a03394d258b471a897b5896cf.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Edge Impulse a Qualcomm company</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand S9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NTc4Nzc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Efinix" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F2f11eecc82be44919dea1c362d5a3509.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Efinix</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Electronics Direct" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fa9796bfa54c244bcaa01338d538f7b7d.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Electronics Direct</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q18</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NTU0NzI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Electronics Weekly" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fce6b9f38140144c9a701b879b53b3f30.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Electronics Weekly</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Media Partner Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NTU0ODU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Electropages" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fe931c15f9f904cee9424763aa78e4896.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Electropages</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Media Partner Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Element Materials Technology" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F191acad2ad1d408fb0714d8bf90f21f0.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Element Materials Technology</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMDI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Elhurt EMS" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F2321f8d76cd44235aa492c5615134b56.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Elhurt EMS</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q19</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Elimo Engineering" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ff4167cf6da4840598d7dbde685dd1214.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Elimo Engineering</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NzQ1NjM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="ELVIN" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fb97c8cbd95c14c86921cf3bd7782eda7.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">ELVIN</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Embedd" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F11fbe2f29e7c40f99b83fef1f6873caf.jpeg&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Embedd</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Embedism" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ffa62f8c94a8d4ae5be595d3e4fee17c0.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Embedism</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="EMC Partner" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F3807f1ad8f4e4b1891e9a1f04e96d5e8.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">EMC Partner</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A22</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjQxOTQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Entech Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F4937dcb9e75044cea5e4e1c0c6cb9f89.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Entech Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Enterprise Recruitment" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd86acd600d5b4e53a59920911445ba25.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Enterprise Recruitment</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Eurofins E&amp;E UK" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F401a87263a7144a498ecb95fbd9bc359.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Eurofins E&amp;E UK</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="European Thermodynamics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc2549e2493b34b858c41e5b8e36f1bc0.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">European Thermodynamics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMDQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Ezurio" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9bb63c2acc5f4219a397fdce00359ee1.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Ezurio</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMTQ0NDc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Farnell Ltd" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F36236dbb38244aadb01112c38d5ae022.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Farnell Ltd</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand L2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="FIGFI" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc31e13643ec741f6a67cdd4d5645c966.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">FIGFI</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NDc3OTY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="FMG Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F4896974ba6d44a95805333a76acd1054.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">FMG Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand S12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Fortec" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ffb664de456084d408809338675819dd9.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Fortec</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjc0MzQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Forward NPD" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd1cbca961b1b40e2a1e43e9619be1aae.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Forward NPD</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMzAzODI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Foundries.io" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7e541d67bddb4a359494d598d68c61e3.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Foundries.io</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand S9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Future Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F40c1233aeb994cef8d51f842de6abc8c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Future Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Garner Osborne Circuits" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F93a7d72dc39e4e658e73d0a0b94539ee.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Garner Osborne Circuits</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand L11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyOTk4ODU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="GB Technology Group" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fe2e14255fa3c40f898e3506a5cbe9cab.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">GB Technology Group</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C15</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="GCT" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F45951162c85949a1a508ea633c263206.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">GCT</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Gelec" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F65ec93223e8547e18e4608c0f74477b7.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Gelec</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Geyer Electronic" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fa5edf71be77e4976b130627ec5908d1c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Geyer Electronic</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Grinn Global" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd0e3ad1cf3a9472a8ba57a47d9764d0d.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Grinn Global</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand L5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Ground Control" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F5fe6611c1973419f82966ed3fc588ccf.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Ground Control</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="GTT Wireless" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F45f4ca2fea4b41abac8cd8215a84ac7c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">GTT Wireless</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MTA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Hamamatsu Photonics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fad43cc442fb24b599368f7486da084f5.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Hamamatsu Photonics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMDY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Hammond Manufacturing" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F6db314271a7745789a462cb8590b6216.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Hammond Manufacturing</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNDE4MTY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Hardware Pioneers" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F748a0866753943a3a5c1ae414202b148.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Hardware Pioneers</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sales Lounge</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Harwin" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fbdd32fa5b9c44c3b988dbc8d2b7cb803.jpeg&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Harwin</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyOTk4ODA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Hectronic" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fb367f57f509043dab5fe801cd234931f.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Hectronic</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0Njk4MzA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Heronic" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fee305696ba804a9c9072fee747e4d0aa.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Heronic</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Startup Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MDk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Hirose Electric Europe" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F13bfba3baa3047e295b57635b190e629.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Hirose Electric Europe</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MTE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Hitaltech" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9282e46035fe438c8222cb9676badf43.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Hitaltech</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Hitex" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc09577e3c0514527845b88a74ecacc51.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Hitex</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F14</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MTI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="i4 Product Design" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fbf7ed570748d42388c264b5739383b11.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">i4 Product Design</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A19</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMDU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="IC Blue" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F37f9b1645d9a4d32904d7e3cee70c814.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">IC Blue</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B19</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MTM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Ignion" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F4b4e0e7409964fc9b13d5baf66a10c4d.jpeg&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Ignion</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MTQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Incap" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F989ca11b975748d6bb64308efca56aed.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Incap</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MTY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Indesmatech" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fdcf53a183d3440d59e59c4fc10c26dae.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Indesmatech</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MTg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Infineon" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F5896c2b779be422f91ba040562cf170b.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Infineon</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F15</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MTk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Innodisk" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F89ebfc3607a94ea99c448e41cb21d68b.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Innodisk</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MjI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Innovative Sensor Technology IST AG" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9c992658f3ef4950942d915303c2cf17.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Innovative Sensor Technology IST AG</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B14</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjQxOTg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="InstaDeep" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F592c8287c11e461aae34633bb3654038.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">InstaDeep</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjQyMDc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Intelligent Group Solutions" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ff432bf18eb2a4003baa43df1581c29bc.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Intelligent Group Solutions</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MzkwMTc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="IoT Insider" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7c3e6826b10e4c0d8ffbc1595e7d6026.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">IoT Insider</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NTU3OTQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="IoT Now" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fbb103ca3423e445988c0b09faff022f5.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">IoT Now</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Media Partner Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NjMzMDM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="IQD Frequency Products" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fb96890d64dab44e9a3013728ed0fcd8a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">IQD Frequency Products</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand S11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MjE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Italtronic" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd23c4941254d459390ec00563ffcd6a3.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Italtronic</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MjA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="J2 Sourcing" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F185eccb1f36b4f25b2596b8c0220fb19.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">J2 Sourcing</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MjQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Jaltek" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F909c368492294eb999f034e9434d3783.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Jaltek</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A17</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MjM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Jauch Quartz" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fb25da9bceb4c4233b1174ffc384347be.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Jauch Quartz</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MjU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Kaizen Technology" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Feeb1829b11054447b8ecf0ca457748d6.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Kaizen Technology</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3Mjg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Keysight Technologies" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F99a1f50a14ab4340abe5a217feb1fc8c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Keysight Technologies</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand L4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MjY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Kigen" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F1af7d08d93b04222aa11cfaa4fd34bfb.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Kigen</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sponsor Only</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3Mjc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Kiwa" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ffaf155352b104e1385ee3adb2c8dc79a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Kiwa</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MzA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="KYOCERA AVX" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc13adc400ecb4b6d9077f208bd7be758.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">KYOCERA AVX</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sponsor Only</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MzE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Laplace Instruments" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F389658cdc1944b4c8295e890f6d5ec06.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Laplace Instruments</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzMDM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Leach" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9d85977cc0084e04a0b48629ce907b63.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Leach</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzNzY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Leonardo" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc0405896958b4e309f9caa5a9bbf6b1a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Leonardo</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B16</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzODM1ODA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Lightbug" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Feeff268ae91f4ffe827762f58baa51f4.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Lightbug</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sponsor Only</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MzI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Linkwave Technologies" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fdf90f55efbe84ad6a7fe2f403ddcf039.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Linkwave Technologies</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzOTE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Liquid Instruments" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F62f5976724bc461f85ee5cac015c0e0b.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Liquid Instruments</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand S4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0Njk4MzU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Literal Labs" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F52aef226e5ec418d80edce04ca29b4f9.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Literal Labs</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Startup Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MzM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="LM Technologies" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F6f02f3ee804d49ea91bbab7adef155f9.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">LM Technologies</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MzQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="LMT IoT" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fbd789d489e914ef69d1222438755cd95.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">LMT IoT</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Luminovo" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F1ec7e625dd7e4cc68c7da72aad311d93.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Luminovo</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="M2MGubbins" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F04543c2716004215884bf45c53570cac.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">M2MGubbins</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjQxOTk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Mandar Solutions" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F961a6e2360484ebeb6c7269b6e71ab4c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Mandar Solutions</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C14</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MzU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Memphis Electronic" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F098663e865c344ffba1d3ff4ae0f40b2.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Memphis Electronic</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3MzY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Micro Crystal" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd45be83441b54824a60f682d5f54d5b7.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Micro Crystal</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3Mzc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Microchip" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd99e155b849b4bd58e6eeda16ccc5ca6.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Microchip</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3Mzg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Midas Displays" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F3cc35c4622a24a66bd6e2833e15440f2.jpeg&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Midas Displays</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NjY4MzY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="MintNeuro" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F843c81f0fd4e4916a72a7ad41768d4cb.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">MintNeuro</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Startup Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMDc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Modnyco" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fdf6d72c91a644c4ab275ed2f53e56f7c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Modnyco</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F17</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NjIwNTM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Molex" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fbb0003d85bcb4b27a3690153f7f6b7a0.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Molex</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3Mzk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Mouser Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F55a085a9873d4bb19a8af6be3721cedb.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Mouser Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDMzODE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Murata" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc2309f8d9db344f9a902b1cfdb7c6913.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Murata</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Nanopower Semiconductor" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fe08567f7dd0b47cc850d92d0a8506140.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Nanopower Semiconductor</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="NCAB Group" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ffd86692bf28549e8ab3d2e64526d3aae.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">NCAB Group</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NTU3OTY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="New Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7d9355cf63fa4c3a975303fdd372ddea.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">New Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Media Partner Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjAyMjQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Newbury Electronics Ltd" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F24eb1e2db2604749a64132b410af80eb.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Newbury Electronics Ltd</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzOTYzOTc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Nex-G" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F910f4f090fc74426b3c09fbbb0221a09.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Nex-G</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI1OTk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="NextPCB" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fde31afdf4b894ceba0d0b9ecb35bb310.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">NextPCB</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Nordic Semiconductor" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fa6163ce4ac2140349edf1e8d1c11977c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Nordic Semiconductor</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyOTk4ODM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="NOTE" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F36a8c99233184c9f972b2371ec8e8630.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">NOTE</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI1OTg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="NotioNext" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F5ba47319d5094aeeb5b07e6c894b240f.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">NotioNext</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A21</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Novocomms" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F4592ff436b11444aa0de117fcef8023c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Novocomms</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjAyMjU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="NTD Shielding" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fcaa1f7f930d64247820921eadfefa386.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">NTD Shielding</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzODU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Nuvonix" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F8e5b15106e3b4249bd5ee2ff445f7147.jpeg&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Nuvonix</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="NXP" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9bd0794fc796490888b5d4f2b66384a2.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">NXP</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="OEM Secrets" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ff58d23481db9482db00110689cfe977a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">OEM Secrets</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sponsor Only</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Panorama Antennas" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ff4216856ab9941ac857981f187de45c3.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Panorama Antennas</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Phase One Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F940cc319ff7344f8806a24319bde55c4.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Phase One Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0Njk4NTY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="PhovIR" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F27e8a91ea5a6403ba37f6a257cf68605.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">PhovIR</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Startup Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MDk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Phytec" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fa0f18fc9a34449129aeeb97989bf946e.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Phytec</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMTQ0NDU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Pico Technology" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F06425751775848a3addd18f685cda141.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Pico Technology</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B18</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Pivot A2E" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fe5d93fb98dba45d79859207a8fad75fb.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Pivot A2E</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDc5OTM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Princeps" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F8e4514f9a9584937833ac56102d26523.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Princeps</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjc0MzM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Printed Systems Limited" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F86a94fca471f4d8399863148cbff87cc.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Printed Systems Limited</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMzAzNzk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Profusion" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F86f33104c5c14fbd947e04ad3ecc9096.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Profusion</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q23</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Prototype Projects" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7bc6f83b6f44455ba94904e0c33f5c27.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Prototype Projects</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Pulsar Measurement" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F4d649fb545254da789239f2c00443267.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Pulsar Measurement</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDMzNzg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="QNX" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7bc8fc7a684a44668e1acb5f8c5f52b2.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">QNX</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzODY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Qualinx" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F76a4e28553ae4da8b8bd7b957a562be5.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Qualinx</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F16</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Quectel" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F78abf0abc94a470fb1bfbde50e6b3007.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Quectel</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MjA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Radiometrix" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F5c97d775298a4fe8a6acdd199601f383.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Radiometrix</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MjE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Raspberry Pi" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9cf73c2227bf4fcc9fc7901faa847be5.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Raspberry Pi</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MjU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Rebels Software" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fcdd3bb8b3d614460b27595bb270049d7.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Rebels Software</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Mjc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Relec Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F48b6a0bf00754431ba9ead577a25b0e5.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Relec Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MjY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Renata Batteries" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F3f8eaa56ee864229bf71c7e15fd879a0.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Renata Batteries</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDMzNzQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Renesas" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F0206a74880854142ae18b349ca31980c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Renesas</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjQxOTc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="RF Solutions" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F8e408719b50847a1b62182c0b31cf421.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">RF Solutions</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MzE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Ripcord Designs" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fd00c63b767b2423198bf89afe2d05095.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Ripcord Designs</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A20</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NzQ1NjQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="RLS" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F99a666c9421c467f95f2672035551635.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">RLS</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MzI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Rohde &amp; Schwarz" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fba9fc45906334a2babffe41dc50214e9.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Rohde &amp; Schwarz</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjc0MzI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="RS" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F4af0d4c17ec147528968d09c1d93acf6.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">RS</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MzM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Rutronik" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fee384836aa524f25b0bcd4a255fca7ae.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Rutronik</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Mzc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="RVL" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F25f60b30aefe41daa305712aca5c4726.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">RVL</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Samtec" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fece6a688dae74163891a1acdc15c4754.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Samtec</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand L1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NzQ1NjU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="SCI Semiconductor" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F60a6c747867f42229c93f7effab499a0.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">SCI Semiconductor</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Seeed Studio" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F293db08c6b404c0cb0d988dbfd210e56.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Seeed Studio</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NTYzNjM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="SeSemi Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F68d443705da8474794bfbe352a07c728.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">SeSemi Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2MTU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Setanta Space" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F8b1fa345c31b4f04941a029afbce48b0.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Setanta Space</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q16</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjAyMzE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="SGS" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F2ea61432e4ee4b6e88ce4d5cc31cf87f.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">SGS</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand S2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzMjc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Shenzhen Grandtop Electronics Co. ,Ltd" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F94bdeb121b734520b1e308723d8cefde.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Shenzhen Grandtop Electronics Co. ,Ltd</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P14</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Mzg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="SiliconExpert" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc78a2d0171ae48cfbb87b476648d13d1.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">SiliconExpert</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMzAzODM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Silvertel" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F43d5305c79f448578af0d19c29bbabe3.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Silvertel</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Mzk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="SIMCom" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ff54a14a02449471abf965bc792101cbf.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">SIMCom</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NDQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Simms International" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fc6e5b9d0622b49dabb3c3f1b6bcfa329.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Simms International</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D7</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NDM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Solid Solutions" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F12ae7af5a768433e8d5296f3881964cc.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Solid Solutions</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NDU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Solsta" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F1cc0d49f0c2d4198a03b507b482607d1.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Solsta</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand L6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NDk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Soracom" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F47bdbbbd8d344634bb9876e050b9f700.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Soracom</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F12</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NTA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Sphere Product Design" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F11ce54fcfed7435fb8306ed242e23586.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Sphere Product Design</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NTE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Starteam Global" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fbb78fa384f5242dfafa250f21a0d2047.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Starteam Global</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMTQ0NDk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="STMicroelectronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F562f235bcbdc4091866998b672908bed.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">STMicroelectronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzMzAzODQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Sundance" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F4fa859af7a194a6cbca3fe7e6b3d62e1.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Sundance</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q20</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NTg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Sunpower Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ffed52a4d657a459c9e990f80a92c7952.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Sunpower Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODExMDA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Syrma SGS" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F2f40e24363824367ac1744ac473d8394.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Syrma SGS</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NjA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="T-Global Technology" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9df4b3aa80e0440aba79f8e58e0c6601.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">T-Global Technology</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyODEwOTU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="TagoIO" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ff38b5185337e4ea6ab6352013d120407.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">TagoIO</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NDExMzY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Taoglas" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F24460e1598524150a85a59041cb59d38.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Taoglas</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NjIwNTA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="TDK" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7e8f5db22a454e0b9420738c1c1064eb.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">TDK</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyOTk4ODE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="TE Connectivity" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F8cdc38c9396f411b851b44ea1c67caa8.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">TE Connectivity</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NjQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Techship" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7b3160cc9a8641feb9eb3284a9828662.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Techship</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NjU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="TeleCANesis" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9a06b2ecf94d4f0b9152a589f19e9a60.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">TeleCANesis</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand N9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNjQxOTY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Teledyne LeCroy" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F2ed847e8dc2a407289d635081851795b.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Teledyne LeCroy</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q22</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NjY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Telit Cinterion" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F56b17c2111cb4a768ed2333040575daf.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Telit Cinterion</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NzA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Telonic Instruments" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fda72981c5d5046c998263375f396a58e.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Telonic Instruments</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand G8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0Mzg5NzY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="The Good Penguin" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F2a810974f9754248a11364731fa2041f.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">The Good Penguin</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NzU4OTQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Tracks Laser &amp; Electronics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Febd3420806124df1b2283fefd154215b.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Tracks Laser &amp; Electronics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A16</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NzI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Trailing Edge Technologies" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F1a12bd4f70b54a3e8ec32c3962377c42.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Trailing Edge Technologies</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand L9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2NzY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Transcend Information" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F2630198c1b6f44f087998c0f0a79cefa.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Transcend Information</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand E4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Nzc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Transfer Multisort Elektronik" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fad7db216449d4d3b86a2e56db4d2232a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Transfer Multisort Elektronik</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand M5</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NjIwNDg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Tria Technologies" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Ff530d0f9b4b44d77a81fb0ebe68da480.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Tria Technologies</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDM5NzU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="TTI" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F70c953bddac343a78620f0ab4ed3ef0c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">TTI</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand L13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2Nzg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="TÜV SÜD" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F15117733c8fd4337ba347b27a86e98ea.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">TÜV SÜD</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDAzNzQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="UK Research and Innovation" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F6368620ae0ca4ff8a7e0908a89df8d15.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">UK Research and Innovation</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A6</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODI="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="UL Solutions" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fb9fcc7dd0b84433f86f585e3b8998709.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">UL Solutions</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P4</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODM="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Uni-T" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F4c520e9d7bd248e9a045bd1bbee5de75.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Uni-T</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Unit 3 Compliance" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F820d4e2fb1904be7bb4c7323d94f41f3.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Unit 3 Compliance</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand F2</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Virscient Limited" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fed3719d5580c442295165ecd4098affb.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Virscient Limited</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand D1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDc5OTQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Virtium" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F66dd36f1a32f4967a85cf82adcca1263.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Virtium</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand P3</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMDg="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="VPG Foil Resistors" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fca68fe3818ea4c12a7dd8b16bb4a0a30.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">VPG Foil Resistors</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand C13</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NjY4MTc="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="weeteq" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F559a05839b974a7f9889f6df84fd1398.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">weeteq</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Startup Zone</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMDk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Winslow Adaptics" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F73b61e7193fb4a31abc157b9cf156a41.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Winslow Adaptics</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand B17</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI3NDk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Wireless Logic" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fe335882f7a7f478db83bd2830296052c.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Wireless Logic</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand L10</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MjE0MDE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Wizerr AI" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F18b5b9189b49414bbbe3f933f8195fca.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Wizerr AI</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R9</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2ODk="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Würth Elektronik" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F61f0e8fe61eb4cfab3edfefb7ce76d1a.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Würth Elektronik</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand S11</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMTA="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Würth Elektronik Circuit Board Technology" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fb6208e82a96544e6b691bbcb0d20f187.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Würth Elektronik Circuit Board Technology</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand A23</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0NjIwNDY="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="XP Power" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F9dbd18f591a14b3eb5652513534a2411.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">XP Power</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand H1</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzI0MDMzNzU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="YOK Energy" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F2107e0a5ffac4a23b1d1a7a86b6b2475.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">YOK Energy</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand R8</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTQ="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Z2Data" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F7292a4db3f174fe7aac725f8dab0bbfd.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Z2Data</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q17</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIzNDMwMTE="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="ZelCom" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2Fcdee4718144a49088f07953bae07275e.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">ZelCom</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Stand Q21</span></div></div></div></div></a><a href="/widget/event/hardware-pioneers-max-26/exhibitor/RXhoaWJpdG9yXzIyNTI2OTU="><div class="sc-ezrbvT dLvqlV sc-jVKzgY gKTVeY"><div class="sc-jdoVuq hSKDIS"><div class="sc-bTJCW ksJpgB sc-VSCQR bFQeus" size="224"><img size="224" ratio="2" class="sc-fZzbfD eYyqii" alt="Zipit Wireless" width="448" height="224" src="https://img.swapcard.com/?o=webp&amp;u=https%3A%2F%2Fstatic.swapcard.com%2Fpublic%2Fimages%2F44a95c98d0bd48078055311ceb19c7aa.png&amp;q=0.8&amp;m=fit&amp;w=448&amp;h=224"></div><div class="sc-hJOweQ bOuHAi"><div class="sc-hrrTzr lgCwUn"><span class="sc-jONoGK sc-dLyDyb hIzuFm kPhijs">Zipit Wireless</span><span class="sc-jONoGK sc-eBLSSh eRbfbq gurUBs">Sponsor Only</span></div></div></div></div></a></div>
"""
BASE_URL = "https://app.swapcard.com"
OUTPUT_CSV = "hardware_pioneers_swapcard_links_extracted.csv"



## 3. Parser Function



In [ ]:
def load_html() -> str:
    if HTML_TEXT.strip():
        return HTML_TEXT

    path = Path(HTML_FILE)
    return path.read_text(encoding="utf-8", errors="ignore")


def parse_swapcard_exhibitor_html(html_text: str, base_url: str = "https://app.swapcard.com") -> list[dict]:
    soup = BeautifulSoup(html_text, "html.parser")
    rows = []
    seen_profile_urls = set()

    location_labels = {
        "startup zone",
        "sponsor only",
        "media partner zone",
        "sales lounge",
        "vip & press lounge",
    }

    for anchor in soup.find_all("a", href=True):
        href = anchor.get("href", "").strip()

        if "/widget/event/" not in href or "/exhibitor/" not in href:
            continue

        profile_url = urljoin(base_url, href)

        if profile_url in seen_profile_urls:
            continue
        seen_profile_urls.add(profile_url)

        image = anchor.find("img")
        company = image.get("alt", "").strip() if image else ""

        spans = [span.get_text(" ", strip=True) for span in anchor.find_all("span")]
        stand_or_location = ""

        for span_text in spans:
            clean = " ".join(span_text.split())
            lower = clean.casefold()

            if re.fullmatch(r"stand\s+[a-z]+\d+", lower) or lower in location_labels:
                stand_or_location = clean
                break

        logo_proxy_url = image.get("src", "").strip() if image else ""
        logo_original_url = ""

        if logo_proxy_url:
            query_params = parse_qs(urlparse(logo_proxy_url).query)
            if "u" in query_params and query_params["u"]:
                logo_original_url = query_params["u"][0]

        if company:
            rows.append({
                "company": company,
                "stand_or_location": stand_or_location,
                "profile_url": profile_url,
                "raw_href": href,
                "logo_proxy_url": logo_proxy_url,
                "logo_original_url": logo_original_url,
            })

    return rows



## 4. Run Extraction



In [ ]:
html_text = load_html()
rows = parse_swapcard_exhibitor_html(html_text, BASE_URL)

print(f"Extracted exhibitors: {len(rows)}")

for row in rows[:10]:
    print(row["company"], "|", row["stand_or_location"], "|", row["profile_url"])



## 5. Export CSV



In [ ]:
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "company",
            "stand_or_location",
            "profile_url",
            "raw_href",
            "logo_proxy_url",
            "logo_original_url",
        ],
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved CSV to: {Path(OUTPUT_CSV).resolve()}")



## 6. Quick Sanity Check



In [ ]:
if rows:
    print("First row:")
    print(rows[0])
else:
    print("No rows found. Check that your HTML contains Swapcard exhibitor card links.")
